## Week-2

In [1]:
import pandas as pd

df = pd.read_csv("CRMLSListing_residential.csv")
df.head(5)

/var/folders/d_/gll3jlg1445_2nm3xgq_xvdc0000gn/T/ipykernel_3099/704966105.py:3: DtypeWarning: Columns (0: ListAgentEmail, 1: BuyerAgencyCompensationType) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("CRMLSListing_residential.csv")


,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,...,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,BuyerOfficeName.1,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,UnparsedAddress.1,source_file
0,1340000.0,1074973329,haleh360@Gmail.com,NaN,NaN,Haleh,Dowlatshahi,34.052207,-118.408445,2220 Avenue Of The Stars 2704,...,False,NaN,NaN,90067,NaN,2105.00,177861.0,NaN,2220 Avenue Of The Stars 2704,CRMLSListing202401.csv
1,2500000.0,1074954552,Reneechen@yourhomesoldguaranteed.com,NaN,NaN,Renee,Chen,33.496363,-117.691677,16 Palisades,...,False,3.0,Capistrano Unified,92677,NaN,254.00,5300.0,NaN,16 Palisades,CRMLSListing202401.csv
2,3150000.0,1074936537,anader@dppre.com,NaN,NaN,Margaret,Nader,34.119345,-118.111254,1615 Waverly Road,...,NaN,2.0,NaN,91108,NaN,NaN,9404.0,NaN,1615 Waverly Road,CRMLSListing202401.csv
3,3090000.0,1074917818,QIANYU0607@GMAIL.COM,NaN,NaN,QIANYU,GUAN,33.984057,-117.802819,2250 Indian Creek Road,...,False,4.0,Walnut Valley Unified,91765,NaN,295.95,58232.0,NaN,2250 Indian Creek Road,CRMLSListing202401.csv
4,12725000.0,1074143166,jeff.williams@pacificsir.com,NaN,NaN,Jeff,Williams,33.607583,-117.887743,317 E. Bayfront,...,False,2.0,Newport Mesa Unified,92662,NaN,0.00,2250.0,NaN,317 E. Bayfront,CRMLSListing202401.csv


In [2]:
num_cols = df.select_dtypes(include=["number"]).columns
cat_cols = df.select_dtypes(exclude=["number"]).columns

print("Numerical columns:", len(num_cols))
print(num_cols)

Numerical columns: 37
Index(['OriginalListPrice', 'ListingKey', 'ClosePrice', 'Latitude',
       'Longitude', 'LivingArea', 'ListPrice', 'DaysOnMarket',
       'FireplacesTotal', 'AboveGradeFinishedArea', 'ListingKeyNumeric',
       'TaxAnnualAmount', 'ParkingTotal', 'LotSizeAcres', 'YearBuilt',
       'DaysOnMarket.1', 'StreetNumberNumeric', 'LivingArea.1',
       'BathroomsTotalInteger', 'BuyerAgencyCompensation', 'TaxYear',
       'BuildingAreaTotal', 'BedroomsTotal', 'Longitude.1',
       'ElementarySchoolDistrict', 'BelowGradeFinishedArea', 'BusinessType',
       'Latitude.1', 'ListPrice.1', 'CoveredSpaces', 'Stories', 'LotSizeArea',
       'MainLevelBedrooms', 'GarageSpaces', 'AssociationFee',
       'LotSizeSquareFeet', 'MiddleOrJuniorSchoolDistrict'],
      dtype='str')


In [3]:
import pandas as pd
import numpy as np
from IPython.display import display


#  Check duplicates
# =========================
total_duplicates = df.duplicated().sum()

if "ListingId" in df.columns:
    listing_duplicates = df.duplicated(subset=["ListingId"]).sum()
else:
    listing_duplicates = None

print("=== DUPLICATE CHECK ===")
print("Total duplicate rows:", total_duplicates)
print("Duplicate ListingId:", listing_duplicates)


# Define core numerical columns
# =========================
num_keep_cols = [
    "ListPrice",
    "ClosePrice",
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "DaysOnMarket",
    "YearBuilt",
    "Latitude",
    "Longitude"
]

# Keep only columns that actually exist in df
num_keep_cols = [col for col in num_keep_cols if col in df.columns]

# =========================
# Generate summary statistics
# =========================
num_cols = df.select_dtypes(include=np.number).columns.tolist()
summary = df[num_cols].describe().T

report = []

for col in num_cols:
    col_data = df[col].dropna()

    missing_count = df[col].isna().sum()
    missing_pct = round(df[col].isna().mean() * 100, 2)

    if col_data.empty:
        report.append([
            col, False,
            missing_count, missing_pct,
            0, 0,
            np.nan, np.nan, np.nan, np.nan,
            np.nan, np.nan,
            0, False, False,
            "Drop"
        ])
        continue

    min_val = summary.loc[col, "min"]
    max_val = summary.loc[col, "max"]
    mean_val = summary.loc[col, "mean"]
    std_val = summary.loc[col, "std"]

    negative_count = int((col_data < 0).sum())
    zero_count = int((col_data == 0).sum())

    # =========================
    #  Detect possible outlier problems
    # =========================

    # Extreme Max
    p99 = col_data.quantile(0.99)
    extreme_max = max_val > p99 * 5 if pd.notna(p99) and p99 != 0 else False

    # High Std
    high_std = std_val > abs(mean_val) * 2 if pd.notna(mean_val) and mean_val != 0 else False

    # IQR Outliers
    q1 = col_data.quantile(0.25)
    q3 = col_data.quantile(0.75)
    iqr = q3 - q1

    if iqr > 0:
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outlier_count = int(((col_data < lower) | (col_data > upper)).sum())
    else:
        outlier_count = 0

    # =========================
    #  Mark whether each column is core
    # =========================
    is_core = col in num_keep_cols

    report.append([
        col, is_core,
        missing_count, missing_pct,
        negative_count, zero_count,
        round(min_val, 2), round(max_val, 2),
        round(mean_val, 2), round(std_val, 2),
        round(q1, 2), round(q3, 2),
        outlier_count, extreme_max, high_std,
        None
    ])

# =========================
# Create report DataFrame
# =========================
report_df = pd.DataFrame(report, columns=[
    "Column", "Is Core",
    "Missing", "Missing %",
    "Negative Count", "Zero Count",
    "Min", "Max", "Mean", "Std",
    "Q1", "Q3",
    "IQR Outliers",
    "Extreme Max ⚠️",
    "High Std ⚠️",
    "Action"
])

# =========================
#  Assign an action
# =========================
def assign_action(row):
    if row["Missing %"] > 90:
        return "Drop"
    elif row["Negative Count"] > 0:
        return "Fix Negative"
    elif row["Extreme Max ⚠️"] or row["High Std ⚠️"] or row["IQR Outliers"] > 0:
        return "Cap / Clean"
    else:
        return "OK"

report_df["Action"] = report_df.apply(assign_action, axis=1)

# =========================
# Sort the results by priority
# =========================
action_order = {
    "Drop": 0,
    "Cap / Clean": 1,
    "Fix Negative": 2,
    "OK": 3
}

report_df["ActionRank"] = report_df["Action"].map(action_order)

report_df = report_df.sort_values(
    by=["ActionRank", "Missing %"],
    ascending=[True, False]
).reset_index(drop=True)

#  Display results in groups
# =========================
print("\n=== DROP ===")
display(report_df[report_df["Action"] == "Drop"])

print("\n=== KEEP (CORE) ===")
display(
    report_df[report_df["Is Core"]]
    .sort_values(by=["ActionRank", "Missing %"])
)

print("\n=== CAP / CLEAN ===")
display(report_df[report_df["Action"] == "Cap / Clean"])

print("\n=== KEEP (OK + Fix Negative) ===")
display(report_df[report_df["Action"].isin(["OK", "Fix Negative"])])

=== DUPLICATE CHECK ===
Total duplicate rows: 0
Duplicate ListingId: 92

=== DROP ===


,Column,Is Core,Missing,Missing %,Negative Count,Zero Count,Min,Max,Mean,Std,Q1,Q3,IQR Outliers,Extreme Max ⚠️,High Std ⚠️,Action,ActionRank
0,FireplacesTotal,False,540183,100.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,False,False,Drop,0
1,AboveGradeFinishedArea,False,540183,100.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,False,False,Drop,0
2,TaxAnnualAmount,False,540183,100.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,False,False,Drop,0
3,TaxYear,False,540183,100.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,False,False,Drop,0
4,ElementarySchoolDistrict,False,540183,100.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,False,False,Drop,0
5,BusinessType,False,540183,100.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,False,False,Drop,0
6,CoveredSpaces,False,540183,100.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,False,False,Drop,0
7,MiddleOrJuniorSchoolDistrict,False,540183,100.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,False,False,Drop,0
8,BelowGradeFinishedArea,False,537154,99.44,0,2439,0.0,5000.0,87.07,360.16,0.0,0.0,0,False,True,Drop,0
9,BuildingAreaTotal,False,492186,91.11,0,192,0.0,123764.0,2445.25,2249.24,1305.0,2800.0,3603,True,False,Drop,0



=== KEEP (CORE) ===


,Column,Is Core,Missing,Missing %,Negative Count,Zero Count,Min,Max,Mean,Std,Q1,Q3,IQR Outliers,Extreme Max ⚠️,High Std ⚠️,Action,ActionRank
25,ListPrice,True,0,0.00,0,0,100.00,195000000.0,1312996.78,2346687.07,580000.00,1375000.00,45490,True,False,Cap / Clean,1
24,BathroomsTotalInteger,True,55,0.01,0,682,0.00,2208.0,2.63,3.26,2.00,3.00,34223,True,False,Cap / Clean,1
23,BedroomsTotal,True,148,0.03,0,2155,0.00,94.0,3.22,1.19,2.00,4.00,1735,True,False,Cap / Clean,1
21,LivingArea,True,556,0.10,0,359,0.00,17021321.0,1980.06,23382.69,1247.00,2300.00,26718,True,True,Cap / Clean,1
18,YearBuilt,True,939,0.17,0,0,1776.00,2028.0,1979.60,26.99,1961.00,2001.00,1365,False,False,Cap / Clean,1
11,ClosePrice,True,394603,73.05,0,0,525.00,820000000.0,1202136.39,4292685.77,600000.00,1350000.00,10480,True,True,Cap / Clean,1
31,DaysOnMarket,True,0,0.00,29,28124,-58.00,731.0,19.54,26.77,5.00,23.00,45168,True,False,Fix Negative,2
27,Latitude,True,80145,14.84,4,60,-117.47,737.0,34.60,2.00,33.74,34.47,92766,True,False,Fix Negative,2
28,Longitude,True,80145,14.84,459893,60,-159.48,329.0,-118.46,3.60,-118.68,-117.24,81070,True,False,Fix Negative,2



=== CAP / CLEAN ===


,Column,Is Core,Missing,Missing %,Negative Count,Zero Count,Min,Max,Mean,Std,Q1,Q3,IQR Outliers,Extreme Max ⚠️,High Std ⚠️,Action,ActionRank
10,BuyerAgencyCompensation,False,461190,85.38,0,220,0.0,2.000000e+05,189.11,2340.72,2.00,2.50,1750,True,True,Cap / Clean,1
11,ClosePrice,True,394603,73.05,0,0,525.0,8.200000e+08,1202136.39,4292685.77,600000.00,1350000.00,10480,True,True,Cap / Clean,1
12,MainLevelBedrooms,False,246075,45.55,0,48071,0.0,3.333000e+03,2.05,6.44,1.00,3.00,805,True,True,Cap / Clean,1
13,AssociationFee,False,129662,24.00,0,175352,0.0,9.683480e+05,270.78,2369.07,0.00,400.00,13117,True,True,Cap / Clean,1
14,LotSizeAcres,False,44518,8.24,0,11831,0.0,4.187292e+06,65.24,12136.81,0.12,0.31,79445,True,True,Cap / Clean,1
15,LotSizeSquareFeet,False,43854,8.12,0,11534,0.0,7.638115e+09,411030.33,20803367.94,5227.00,13615.00,79560,True,True,Cap / Clean,1
16,LotSizeArea,False,43679,8.09,0,11552,0.0,9.187423e+08,46953.52,2178789.61,5000.00,12051.00,76300,True,True,Cap / Clean,1
17,GarageSpaces,False,29991,5.55,0,66816,0.0,6.000000e+02,1.85,3.88,1.00,2.00,12394,True,True,Cap / Clean,1
18,YearBuilt,True,939,0.17,0,0,1776.0,2.028000e+03,1979.60,26.99,1961.00,2001.00,1365,False,False,Cap / Clean,1
19,StreetNumberNumeric,False,850,0.16,0,345,0.0,7.954796e+07,11533.52,281856.83,975.00,12060.00,50709,True,True,Cap / Clean,1



=== KEEP (OK + Fix Negative) ===


,Column,Is Core,Missing,Missing %,Negative Count,Zero Count,Min,Max,Mean,Std,Q1,Q3,IQR Outliers,Extreme Max ⚠️,High Std ⚠️,Action,ActionRank
27,Latitude,True,80145,14.84,4,60,-1.174700e+02,7.370000e+02,3.460000e+01,2.00,3.374000e+01,3.447000e+01,92766,True,False,Fix Negative,2
28,Longitude,True,80145,14.84,459893,60,-1.594800e+02,3.290000e+02,-1.184600e+02,3.60,-1.186800e+02,-1.172400e+02,81070,True,False,Fix Negative,2
29,Longitude.1,False,80145,14.84,459893,60,-1.594800e+02,3.290000e+02,-1.184600e+02,3.60,-1.186800e+02,-1.172400e+02,81070,True,False,Fix Negative,2
30,Latitude.1,False,80145,14.84,4,60,-1.174700e+02,7.370000e+02,3.460000e+01,2.00,3.374000e+01,3.447000e+01,92766,True,False,Fix Negative,2
31,DaysOnMarket,True,0,0.00,29,28124,-5.800000e+01,7.310000e+02,1.954000e+01,26.77,5.000000e+00,2.300000e+01,45168,True,False,Fix Negative,2
32,ParkingTotal,False,20,0.00,143,42883,-1.430000e+02,2.593669e+06,7.620000e+00,3529.17,2.000000e+00,3.000000e+00,91517,True,True,Fix Negative,2
33,DaysOnMarket.1,False,0,0.00,29,28124,-5.800000e+01,7.310000e+02,1.954000e+01,26.77,5.000000e+00,2.300000e+01,45168,True,False,Fix Negative,2
34,Stories,False,94249,17.45,0,0,1.000000e+00,2.000000e+00,1.370000e+00,0.48,1.000000e+00,2.000000e+00,0,False,False,OK,3
35,ListingKey,False,0,0.00,0,0,1.018357e+09,1.157178e+09,1.104799e+09,29883843.49,1.077379e+09,1.126880e+09,0,False,False,OK,3
36,ListingKeyNumeric,False,0,0.00,0,0,1.018357e+09,1.157178e+09,1.104799e+09,29883843.49,1.077379e+09,1.126880e+09,0,False,False,OK,3


##Second Screening: "CAP / CLEAN" Section and "KEEP (OK + Fix Negative)" Section

Drop: BuyerAgencyCompensation, AssociationFee, StreetNumberNumeric,ListingKey,ListingKeyNumeric	

duplicate(Needs to be deleted): LivingArea.1, ListPrice.1,Latitude.1,Longitude.1,DaysOnMarket.1

keep: 
BuildingAreaTotal-Total area of the structure. 

ClosePrice	

MainLevelBedrooms

GarageSpaces

BedroomsTotal

LotSizeSquareFeet

LotSizeArea

BathroomsTotalInteger

YearBuilt

OriginalListPrice

ListPrice	


